# 02 Aggregate scalings and calculate derivatives

In [6]:
import pandas as pd
import numpy as np
import warnings

## Load scalings

In [2]:
scalings = {cond: {sis: pd.read_parquet(f"{cond}/{cond}.{sis}.scaling.parquet")
                   for sis in ('cis', 'trans')}
            for cond in ('WT_G2','dCTCF_G2', 'dNIPBL_G2', 'dWAPL_G2')}

## Aggregate scalings and calculate derivatives, save to files

In [3]:
def agg_scaling(data):
    agg_data = data.groupby(['min_dist','max_dist'])\
                   .agg({'n_pairs':'sum', 'n_bp2':'sum'})\
                   .reset_index()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        dist_bin_mids = np.sqrt(agg_data['min_dist'] * agg_data['max_dist'])
    pair_freqs = agg_data['n_pairs'] / agg_data['n_bp2']
    mask = pair_freqs > 0
    norm_freqs = pair_freqs / np.sum(pair_freqs[mask])
    total_data = pd.DataFrame({'dist': dist_bin_mids,
                               'freq': pair_freqs,
                               'norm_freq': norm_freqs,
                               'mask': mask,
                               'min_dist': agg_data['min_dist'],
                               'max_dist': agg_data['max_dist'],
                               'n_pairs': agg_data['n_pairs']})
    return total_data

In [8]:
def calc_derivative(agg_data):
    pair_freqs = agg_data['freq']
    dist_bin_mids = agg_data['dist']
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        pair_freqs_derivative_log10 = np.diff(np.log10(pair_freqs.values)) / np.diff(np.log10(dist_bin_mids.values))
    derivative_support = np.sqrt(dist_bin_mids.values[1:] * dist_bin_mids.values[:-1])
    derivative_data = pd.DataFrame({'dist': derivative_support, 'd_freq_log10': pair_freqs_derivative_log10})
    return derivative_data

In [9]:
for cond, sis_data in scalings.items():
    for sis, data in sis_data.items():
        agg_data = agg_scaling(data)
        derivative = calc_derivative(agg_data)
        agg_data.to_parquet(f"{cond}/{cond}.{sis}.agg_scaling.parquet")
        derivative.to_parquet(f"{cond}/{cond}.{sis}.derivative.parquet")